# Figure 01 -- force error vs expansion order

Loads `results/validation/force_error_vs_order.json`, produced by `bench/validation/force_error_vs_order.py`. No computation here.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == "jaccpot_paper" else pathlib.Path.cwd()))

import numpy as np
import matplotlib.pyplot as plt

from examples.jaccpot_paper.common import jsonio, style

style.apply()
FIG_DIR = jsonio.repo_root() / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
art = jsonio.read_result("validation/force_error_vs_order.json")
cfg, recs = art["config"], art["data"]["records"]

fig, axes = style.figure(width=style.TWO_COL, height=2.7, ncols=2, sharex=True)
bases = [b for b in cfg["basis"]]
dists = [d for d in cfg["distribution"]]
# Dash pattern carries the distribution and colour carries the basis, so the two
# axes of the sweep are never confused for one another.
dashes = {d: ls for d, ls in zip(dists, ["-", "--", ":"])}

for ax, field, label in (
    (axes[0], "rel_l2", "relative $L_2$ force error"),
    (axes[1], "worst_component", "worst component error / rms$|a|$"),
):
    for bi, basis in enumerate(bases):
        for dist in dists:
            sel = sorted(
                (r for r in recs if r["basis"] == basis and r["distribution"] == dist),
                key=lambda r: r["order"],
            )
            if not sel:
                continue
            ax.plot(
                [r["order"] for r in sel],
                [r[field] for r in sel],
                marker=style.MARKERS[bi % len(style.MARKERS)],
                linestyle=dashes[dist],
                color=style.entity_color(basis, bi),
                label=f"{basis}, {dist}",
                markerfacecolor="white",
                markeredgecolor=style.entity_color(basis, bi),
            )
    ax.set_yscale("log")
    ax.set_xlabel("expansion order $p$")
    ax.set_ylabel(label)
    style.finish(ax, legend=(ax is axes[0]), legend_kwargs={"loc": "lower left"})

style.annotate_config(
    axes[1],
    jsonio.config_caption(cfg, ["n", "theta", "leaf_size", "preset", "precision", "device"]),
    loc="upper right",
)
fig.tight_layout()
style.save(fig, FIG_DIR / "fig01_force_error_vs_order.pdf")


## Caption

Force error against an exact direct summation as a function of expansion order
$p$, at fixed $N$ and opening angle, for the real (Dehnen) and solidfmm bases on
a uniform cube and a Plummer sphere. **Left:** relative $L_2$ error over the whole
acceleration field. **Right:** the largest single-component error, normalised by
the rms $|a|$ -- reported alongside the $L_2$ norm because an $L_2$ norm averages
away the tail that actually limits an integrator. Exact configuration is
annotated; all values are read from
`results/validation/force_error_vs_order.json`.
